In [ ]:
# Convert MTBS shapefile to Geodatabase feature class
import os
import arcpy

# Initialize script workspace
arcpy.env.overwriteOutput = True

base_dir = os.path.dirname(os.path.abspath("__file__"))

# Define input/target paths relative to repository data directory
in_shapefile = os.path.join(base_dir, "data", "mtbs_perimeter_data", "mtbs_perims_DD.shp")
target_gdb   = os.path.join(base_dir, "data", "mtbs_perimeter_data", "MTBS.gdb")
output_name  = "MTBS_perims"

out_fc = os.path.join(target_gdb, output_name)

print("Exporting MTBS shapefile to Geodatabase...")

# Export shapefile to Geodatabase feature class
arcpy.conversion.ExportFeatures(
    in_features=in_shapefile,
    out_features=out_fc
)

print(f"Successfully created {output_name} inside MTBS.gdb.")

In [ ]:
# 1994–2024 MTBS Date Filter
import os
import arcpy

# Overwrite safeguard
arcpy.env.overwriteOutput = True

# Define base path
base_dir = os.path.dirname(os.path.abspath("__file__"))

# --- INPUTS & OUTPUTS ---
gdb_path   = os.path.join(base_dir, "data", "mtbs_perimeter_data", "MTBS.gdb")
fc         = os.path.join(gdb_path, "MTBS_perims")
layer_name = "MTBS_lyr"
out_fc     = os.path.join(gdb_path, "MTBS_1994_2024")

# --- 1. Make a feature layer ---
arcpy.management.MakeFeatureLayer(fc, layer_name)

# --- 2. Apply the date filter ---
query = "Ig_Date >= DATE '1994-01-01' AND Ig_Date <= DATE '2024-12-31'"
arcpy.management.SelectLayerByAttribute(layer_name, "NEW_SELECTION", query)

# Check how many features were selected
match_count = int(arcpy.management.GetCount(layer_name)[0])
print(f"Applied date filter. Found {match_count:,} matching MTBS perimeters.")

# --- 3. Export the filtered subset ---
arcpy.management.CopyFeatures(layer_name, out_fc)

print(f"Exported filtered MTBS subset to: {out_fc}")

In [ ]:
# Project MTBS data to match SEFM coordinate system
import os
import arcpy

arcpy.env.overwriteOutput = True

# Define base path
base_dir = os.path.dirname(os.path.abspath("__file__"))

# --- INPUTS & OUTPUTS ---
gdb_path = os.path.join(base_dir, "data", "mtbs_perimeter_data", "MTBS.gdb")
in_fc    = os.path.join(gdb_path, "MTBS_1994_2024")
out_fc   = os.path.join(gdb_path, "MTBS_1994_2024_projected")

# Path to master layer to pull the target spatial reference template
sefm_master = os.path.join(base_dir, "output", "ClassiFIRE.gdb", "SEFM_events_94_24")

print("Retrieving spatial reference from SEFM master dataset...")
target_spatial_ref = arcpy.Describe(sefm_master).spatialReference
print(f"Target Projection identified as: {target_spatial_ref.name}")

print("Projecting MTBS data...")
# Syntax: Project(in_dataset, out_dataset, out_coor_system, {transform_method})
arcpy.management.Project(
    in_fc,
    out_fc,
    target_spatial_ref,
    "NAD_1983_To_WGS_1984_5"
)

print(f"Projected MTBS dataset saved to: {out_fc}")

In [ ]:
# clip to extent of SEFM
import arcpy

# --- INPUTS & OUTPUTS ---
mtbs_gdb = os.path.join(base_dir, "data", "mtbs_perimeter_data", "MTBS.gdb")
mtbs_fc  = os.path.join(mtbs_gdb, "MTBS_1994_2024_projected")
sefm_fc  = os.path.join(base_dir, "output", "ClassiFIRE.gdb", "extent_Dissolved")
out_fc   = os.path.join(mtbs_gdb, "MTBS_1994_2024_projected_clipped")

arcpy.env.overwriteOutput = True

print("Clipping MTBS perimeters to SEFM extent...")

# Modern alternative using safe positional arguments
# Syntax: Clip(in_features, clip_features, out_feature_class)
arcpy.analysis.Clip(
    mtbs_fc,
    sefm_fc,
    out_fc
)

print(f"Successfully clipped MTBS to SEFM extent: {out_fc}")